# Nexus — Supplier Performance Analysis
## Outputs all metrics the dashboard Supplier page reads from
> Reliability scoring, lead time variance, fulfillment rate, stockout attribution per supplier

## Section 0 — Imports & Config

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

DATA_DIR   = Path('/Users/shashi/retail-ai-project/data/raw/output/csv')
OUTPUT_DIR = Path('/Users/shashi/retail-ai-project/data/processed/nexus/supplier')
OUTPUT_DIR.mkdir(exist_ok=True)

NOW = datetime.now()
print(f'Supplier Performance Analysis — {NOW.strftime("%B %d, %Y %I:%M %p")}')


Supplier Performance Analysis — April 25, 2026 01:27 PM


## Section 1 — Load Data

In [2]:
stores    = pd.read_csv(DATA_DIR / 'stores.csv')
products  = pd.read_csv(DATA_DIR / 'products.csv')
suppliers = pd.read_csv(DATA_DIR / 'suppliers.csv', parse_dates=['contract_start'])
stockout  = pd.read_csv(DATA_DIR / 'stockout_events.csv',
                          parse_dates=['stockout_date','restock_date'])

print('Loading replenishment logs in chunks...')
repl_chunks = []
for chunk in pd.read_csv(DATA_DIR / 'replenishment_logs.csv',
                          parse_dates=['order_date','receive_date'],
                          chunksize=500_000):
    repl_chunks.append(chunk)
repl = pd.concat(repl_chunks, ignore_index=True)

# Attach supplier_id to replenishment via products
prod_sup = products[['sku_id','supplier_id','category','unit_price']]
repl = repl.merge(prod_sup, on='sku_id', how='left')
repl = repl.merge(suppliers[['supplier_id','supplier_name','lead_time_days_avg',
                               'lead_time_days_std','reliability_score']],
                  on='supplier_id', how='left')

# Attach supplier to stockouts
stockout = stockout.merge(prod_sup, on='sku_id', how='left')
stockout = stockout.merge(suppliers[['supplier_id','supplier_name']], on='supplier_id', how='left')
stockout = stockout.merge(stores[['store_id','region','store_format']], on='store_id', how='left')

print(f'Replenishment rows : {len(repl):,}')
print(f'Stockout events    : {len(stockout):,}')
print(f'Suppliers          : {len(suppliers):,}')


Loading replenishment logs in chunks...
Replenishment rows : 1,366,761
Stockout events    : 313,599
Suppliers          : 85


## Section 2 — Fulfillment & Lead Time Metrics per Supplier

In [3]:
repl['fulfillment_rate'] = (repl['units_received'] /
                             repl['units_ordered'].replace(0, np.nan)).clip(0, 1)
repl['lt_vs_promised']   = repl['lead_time_actual'] - repl['lead_time_days_avg']
repl['is_late']          = repl['lt_vs_promised'] > 0
repl['is_short']         = repl['fulfillment_rate'] < 0.95

sup_metrics = (
    repl.groupby('supplier_id')
    .agg(
        total_orders         = ('replenishment_id',  'count'),
        avg_lead_actual      = ('lead_time_actual',  'mean'),
        std_lead_actual      = ('lead_time_actual',  'std'),
        avg_fulfillment_rate = ('fulfillment_rate',  'mean'),
        late_delivery_rate   = ('is_late',           'mean'),
        short_delivery_rate  = ('is_short',          'mean'),
        total_units_ordered  = ('units_ordered',     'sum'),
        total_units_received = ('units_received',    'sum'),
        total_repl_cost      = ('replenishment_cost','sum'),
        avg_lt_vs_promised   = ('lt_vs_promised',    'mean'),
        stores_served        = ('store_id',          'nunique'),
        skus_supplied        = ('sku_id',            'nunique'),
    )
    .reset_index()
)
sup_metrics['std_lead_actual'] = sup_metrics['std_lead_actual'].fillna(0)

# Stockout attribution per supplier
sup_stockout = (
    stockout[stockout['root_cause'].isin(['Supplier delay','Supplier Delay'])]
    .groupby('supplier_id')
    .agg(
        stockout_events_caused  = ('stockout_id',             'count'),
        stockout_revenue_lost   = ('estimated_lost_revenue',  'sum'),
        avg_stockout_duration   = ('duration_days',           'mean'),
    )
    .reset_index()
)

# All stockout events (not just supplier-caused)
sup_all_stockout = (
    stockout.groupby('supplier_id')
    .agg(
        total_stockout_events   = ('stockout_id',             'count'),
        total_revenue_at_risk   = ('estimated_lost_revenue',  'sum'),
    )
    .reset_index()
)

# Merge everything into one supplier scorecard
scorecard = (
    suppliers
    .merge(sup_metrics,       on='supplier_id', how='left')
    .merge(sup_stockout,      on='supplier_id', how='left')
    .merge(sup_all_stockout,  on='supplier_id', how='left')
)

# Fill nulls
for col in ['stockout_events_caused','stockout_revenue_lost',
            'total_stockout_events','total_revenue_at_risk']:
    scorecard[col] = scorecard[col].fillna(0)

# Composite risk score (0-100, higher = more risky)
def minmax(s):
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else pd.Series(0.5, index=s.index)

scorecard['risk_score'] = (
    minmax(scorecard['late_delivery_rate'].fillna(0))   * 0.30 +
    minmax(scorecard['short_delivery_rate'].fillna(0))  * 0.25 +
    minmax(scorecard['std_lead_actual'].fillna(0))      * 0.20 +
    minmax(scorecard['total_stockout_events'].fillna(0))* 0.25
) * 100

scorecard['risk_tier'] = pd.cut(
    scorecard['risk_score'],
    bins=[-1, 33, 66, 101],
    labels=['LOW_RISK', 'MEDIUM_RISK', 'HIGH_RISK']
)

print(f'Supplier scorecard rows: {len(scorecard)}')
print(f"HIGH_RISK suppliers    : {(scorecard['risk_tier']=='HIGH_RISK').sum()}")
print(f"LOW_RISK suppliers     : {(scorecard['risk_tier']=='LOW_RISK').sum()}")


Supplier scorecard rows: 85
HIGH_RISK suppliers    : 2
LOW_RISK suppliers     : 28


## Section 3 — Category-Level Supplier Risk

In [4]:
# Which categories are most exposed to supplier risk?
cat_sup_risk = (
    repl.groupby(['supplier_id','category'])
    .agg(
        orders           = ('replenishment_id', 'count'),
        avg_fulfillment  = ('fulfillment_rate', 'mean'),
        late_rate        = ('is_late',          'mean'),
    )
    .reset_index()
    .merge(suppliers[['supplier_id','supplier_name','reliability_score']],
           on='supplier_id', how='left')
)

cat_exposure = (
    cat_sup_risk.groupby('category')
    .agg(
        avg_fulfillment  = ('avg_fulfillment', 'mean'),
        avg_late_rate    = ('late_rate',        'mean'),
        supplier_count   = ('supplier_id',      'nunique'),
    )
    .reset_index()
    .sort_values('avg_late_rate', ascending=False)
)

print('CATEGORY SUPPLIER EXPOSURE (worst first):')
print(cat_exposure.to_string(index=False))


CATEGORY SUPPLIER EXPOSURE (worst first):
         category  avg_fulfillment  avg_late_rate  supplier_count
    Personal Care           0.9218         0.2506              61
           Bakery           0.9209         0.2502              63
     Dairy & Eggs           0.9216         0.2375              69
           Snacks           0.9213         0.2299              66
     Frozen Foods           0.9214         0.2287              61
     Pet Supplies           0.9214         0.2262              46
   Meat & Seafood           0.9214         0.2204              57
  Canned & Pantry           0.9212         0.2193              51
    Baby & Infant           0.9220         0.2169              45
          Produce           0.9218         0.2035              53
        Household           0.9208         0.2000              54
        Beverages           0.9203         0.1872              62
Health & Wellness           0.9206         0.1823              41


## Section 4 — Store-Level Supplier Impact

In [5]:
store_sup_impact = (
    repl.groupby(['store_id','supplier_id'])
    .agg(
        orders           = ('replenishment_id', 'count'),
        avg_fulfillment  = ('fulfillment_rate', 'mean'),
        late_rate        = ('is_late',          'mean'),
        total_cost       = ('replenishment_cost','sum'),
    )
    .reset_index()
    .merge(suppliers[['supplier_id','supplier_name','reliability_score']],
           on='supplier_id', how='left')
    .merge(stores[['store_id','store_name','region']], on='store_id', how='left')
)

# Worst store-supplier combos
worst_combos = store_sup_impact.nlargest(20, 'late_rate')
print('TOP 20 WORST STORE-SUPPLIER COMBINATIONS (by late delivery rate):')
print(worst_combos[['store_id','store_name','supplier_name',
                     'orders','avg_fulfillment','late_rate','total_cost']]
      .to_string(index=False))


TOP 20 WORST STORE-SUPPLIER COMBINATIONS (by late delivery rate):
store_id             store_name         supplier_name  orders  avg_fulfillment  late_rate   total_cost
   S0001   GreenCorner New York      CoastalTrade LLC      24           0.9103     1.0000 322,965.0000
   S0001   GreenCorner New York       FairDeal Supply      26           0.9401     1.0000  58,058.2000
   S0001   GreenCorner New York    ZealandImports LLC      40           0.9313     1.0000 136,579.0500
   S0002 SummitMart Los Angeles   GlobalReach Trading      60           0.9274     1.0000 255,625.2300
   S0002 SummitMart Los Angeles SunState Distributors      53           0.9235     1.0000 241,131.5000
   S0002 SummitMart Los Angeles    VanguardSupply Co.      21           0.9350     1.0000  31,476.8700
   S0003   SummitCorner Chicago YellowBrick Wholesale      50           0.9291     1.0000  68,020.0000
   S0003   SummitCorner Chicago  XpertChain Logistics      54           0.9255     1.0000 406,564.8600
   S000

## Section 5 — Visualizations

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'Supplier Performance Scorecard — {NOW.strftime("%B %d, %Y")}',
             fontsize=13, fontweight='bold', color='white')
fig.patch.set_facecolor('#0D1B2A')
for ax in axes.flat:
    ax.set_facecolor('#132238')
    ax.tick_params(colors='white')
    ax.spines['bottom'].set_color('#2D4A6A')
    ax.spines['left'].set_color('#2D4A6A')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# P1: Reliability score distribution
axes[0,0].hist(scorecard['reliability_score'].dropna(), bins=20,
               color='#0D9488', edgecolor='white', alpha=0.85)
axes[0,0].axvline(0.90, color='#EF4444', linestyle='--', label='90% threshold')
axes[0,0].set_title('Reliability Score Distribution', color='white', fontweight='bold')
axes[0,0].set_xlabel('Reliability Score', color='#94A3B8')
axes[0,0].legend(labelcolor='white')

# P2: Late delivery rate — worst 15 suppliers
worst15 = scorecard.nlargest(15, 'late_delivery_rate').dropna(subset=['late_delivery_rate'])
colors2  = ['#EF4444' if v > 0.3 else '#F59E0B' if v > 0.15 else '#10B981'
             for v in worst15['late_delivery_rate']]
axes[0,1].barh(worst15['supplier_name'].str[:20], worst15['late_delivery_rate']*100,
               color=colors2)
axes[0,1].set_title('Late Delivery Rate — Worst 15', color='white', fontweight='bold')
axes[0,1].set_xlabel('Late Rate (%)', color='#94A3B8')

# P3: Fulfillment rate vs reliability score scatter
sc = axes[1,0].scatter(
    scorecard['reliability_score'],
    scorecard['avg_fulfillment_rate'],
    c=scorecard['risk_score'],
    cmap='RdYlGn_r', s=60, alpha=0.8
)
plt.colorbar(sc, ax=axes[1,0], label='Risk Score')
axes[1,0].set_title('Reliability vs Fulfillment Rate', color='white', fontweight='bold')
axes[1,0].set_xlabel('Reliability Score', color='#94A3B8')
axes[1,0].set_ylabel('Avg Fulfillment Rate', color='#94A3B8')

# P4: Risk tier pie
tier_counts = scorecard['risk_tier'].value_counts()
axes[1,1].pie(tier_counts.values,
              labels=tier_counts.index,
              colors=['#10B981','#F59E0B','#EF4444'],
              autopct='%1.1f%%', startangle=90,
              textprops={'color':'white'},
              wedgeprops={'edgecolor':'#0D1B2A','linewidth':2})
axes[1,1].set_title('Supplier Risk Tier Distribution', color='white', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'supplier_performance.png', bbox_inches='tight',
            dpi=130, facecolor='#0D1B2A')
plt.show()
print('Saved: supplier_performance.png')


Saved: supplier_performance.png


## Section 6 — Export All Outputs

In [7]:
scorecard.insert(0, 'report_time', NOW.strftime('%Y-%m-%d %H:%M:%S'))
scorecard.to_csv(OUTPUT_DIR / 'supplier_scorecard.csv', index=False)
print(f'supplier_scorecard.csv       — {len(scorecard)} suppliers')

cat_exposure.insert(0, 'report_time', NOW.strftime('%Y-%m-%d %H:%M:%S'))
cat_exposure.to_csv(OUTPUT_DIR / 'supplier_category_exposure.csv', index=False)
print(f'supplier_category_exposure.csv — {len(cat_exposure)} categories')

worst_combos.to_csv(OUTPUT_DIR / 'worst_store_supplier_combos.csv', index=False)
print(f'worst_store_supplier_combos.csv — {len(worst_combos)} combos')

store_sup_impact.to_csv(OUTPUT_DIR / 'store_supplier_impact.csv', index=False)
print(f'store_supplier_impact.csv    — {len(store_sup_impact):,} rows')

print()
print('='*56)
print('  SUPPLIER PERFORMANCE SUMMARY')
print('='*56)
print(f'  Total suppliers        : {len(scorecard)}')
print(f'  HIGH RISK              : {(scorecard["risk_tier"]=="HIGH_RISK").sum()}')
print(f'  MEDIUM RISK            : {(scorecard["risk_tier"]=="MEDIUM_RISK").sum()}')
print(f'  LOW RISK               : {(scorecard["risk_tier"]=="LOW_RISK").sum()}')
print(f'  Avg fulfillment rate   : {scorecard["avg_fulfillment_rate"].mean():.1%}')
print(f'  Avg late delivery rate : {scorecard["late_delivery_rate"].mean():.1%}')
print(f'  Total stockout rev lost: ${scorecard["total_revenue_at_risk"].sum():,.0f}')
print('='*56)


supplier_scorecard.csv       — 85 suppliers
supplier_category_exposure.csv — 13 categories
worst_store_supplier_combos.csv — 20 combos
store_supplier_impact.csv    — 24,738 rows

  SUPPLIER PERFORMANCE SUMMARY
  Total suppliers        : 85
  HIGH RISK              : 2
  MEDIUM RISK            : 55
  LOW RISK               : 28
  Avg fulfillment rate   : 92.0%
  Avg late delivery rate : 21.1%
  Total stockout rev lost: $459,954,660
